# Notebook 2 — Cleaning the data and storing it in MongoDB

## What we do
1. Read the raw JSON and build a **table** (a Pandas DataFrame) with one row per film.
2. **Check the quality** of the data: missing values, duplicates, strange values.
3. **Clean** it.
4. Save a clean CSV file.
5. Store the films in **MongoDB** and run some queries.

## Why cleaning matters
A machine learning model learns from the data it receives. If the data contains errors, the model learns the errors ("garbage in, garbage out").

## A special trap in TMDB
TMDB writes `budget = 0` when the budget is **unknown**. A budget of 0 dollars is not real, so we replace it with "missing" (`NaN`).

In [1]:
import os

# The notebooks live in notebooks/, but the data folder is at the project root.
# This moves us to the project root so paths like "data/raw/..." always work.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working folder:", os.getcwd())

Working folder: c:\Users\ycode\Desktop\Projet\Machine Learning\CineMatch AI


In [2]:
import json
import os

import numpy as np
import pandas as pd

## 1. Read the raw JSON and build a table

In [3]:
with open("data/raw/tmdb_raw.json", encoding="utf-8") as file:
    raw = json.load(file)

movies = list(raw["movie_details"].values())
print("Number of films in the JSON:", len(movies))

Number of films in the JSON: 3000


In the JSON, `genres` looks like `[{"id": 28, "name": "Action"}, ...]`. We only want the **names**, so we build a simple list of names. Same idea for the keywords.

In [4]:
rows = []
for movie in movies:
    genres = []
    for g in movie.get("genres", []):
        genres.append(g["name"])

    keywords = []
    for k in movie.get("keywords", {}).get("keywords", []):
        keywords.append(k["name"])

    rows.append({
        "movie_id": movie.get("id"),
        "title": movie.get("title"),
        "overview": movie.get("overview"),
        "release_date": movie.get("release_date"),
        "runtime": movie.get("runtime"),
        "original_language": movie.get("original_language"),
        "genres": genres,
        "keywords": keywords,
        "budget": movie.get("budget"),
        "revenue": movie.get("revenue"),
        "popularity": movie.get("popularity"),
        "vote_average": movie.get("vote_average"),
        "vote_count": movie.get("vote_count"),
    })

df = pd.DataFrame(rows)
print(df.shape)
df.head()

(3000, 13)


,movie_id,title,overview,release_date,runtime,original_language,genres,keywords,budget,revenue,popularity,vote_average,vote_count
0,969681,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,2026-07-29,145,en,"[Science Fiction, Action, Adventure]","[mind control, new york city, hero, mutation, ...",225000000,2505477000,623.3942,7.900,3022
1,1599191,Renegade Immortal: Battle of the Immortal Slayer,Trapped in a fatal pursuit within the Celestia...,2026-10-01,0,zh,"[Animation, Action, Adventure, Fantasy]","[based on fairy tale, xianxia, based on tv ser...",0,0,432.5374,7.800,12
2,1423191,Resident Evil,Medical courier Bryan unwittingly finds himsel...,2026-09-16,95,en,"[Horror, Science Fiction, Adventure]","[outbreak, survival, decapitation, zombie, bas...",75000000,245400000,364.5221,7.299,750
3,1368337,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",2026-07-15,173,en,"[Adventure, Action, Fantasy]","[ship, trojan war, greek mythology, historical...",250000000,1767464000,322.0423,8.012,4016
4,1228834,The Fix,Disillusioned by the end of the war in Afghani...,2026-09-10,142,en,"[Action, Thriller]","[central intelligence agency (cia), war vetera...",32000000,0,319.3784,7.100,46


## 2. Quality check (before cleaning)

In [5]:
df.info()          # column types and number of non-empty values

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movie_id           3000 non-null   int64  
 1   title              3000 non-null   str    
 2   overview           3000 non-null   str    
 3   release_date       3000 non-null   str    
 4   runtime            3000 non-null   int64  
 5   original_language  3000 non-null   str    
 6   genres             3000 non-null   object 
 7   keywords           3000 non-null   object 
 8   budget             3000 non-null   int64  
 9   revenue            3000 non-null   int64  
 10  popularity         3000 non-null   float64
 11  vote_average       3000 non-null   float64
 12  vote_count         3000 non-null   int64  
dtypes: float64(2), int64(5), object(2), str(4)
memory usage: 1.1+ MB


In [6]:
print("Missing values per column:")
print(df.isna().sum())

print("\nDuplicated movie_id:", df.duplicated("movie_id").sum())
print("Duplicated title + date:", df.duplicated(["title", "release_date"]).sum())

Missing values per column:
movie_id             0
title                0
overview             0
release_date         0
runtime              0
original_language    0
genres               0
keywords             0
budget               0
revenue              0
popularity           0
vote_average         0
vote_count           0
dtype: int64

Duplicated movie_id: 0
Duplicated title + date: 0


In [7]:
print("Suspicious values:")
print("budget = 0       :", (df["budget"] == 0).sum())
print("revenue = 0      :", (df["revenue"] == 0).sum())
print("runtime = 0      :", (df["runtime"] == 0).sum())
print("empty overview   :", (df["overview"] == "").sum())
print("vote_count = 0   :", (df["vote_count"] == 0).sum())

df.describe()

Suspicious values:
budget = 0       : 614
revenue = 0      : 564
runtime = 0      : 35
empty overview   : 12
vote_count = 0   : 51


,movie_id,runtime,budget,revenue,popularity,vote_average,vote_count
count,3.000000e+03,3000.000000,3.000000e+03,3.000000e+03,3000.000000,3000.000000,3000.000000
mean,4.119694e+05,109.525000,4.478020e+07,1.515213e+08,18.756856,6.690375,4157.432667
std,4.996510e+05,26.504481,5.862273e+07,2.539242e+08,25.532677,1.211117,4990.735793
min,1.100000e+01,0.000000,0.000000e+00,0.000000e+00,5.645100,0.000000,0.000000
25%,9.765000e+03,96.000000,2.686582e+06,4.807341e+06,10.336350,6.260250,1050.500000
50%,1.584315e+05,107.500000,2.395000e+07,6.123829e+07,12.832550,6.800000,2609.500000
75%,7.009245e+05,123.000000,6.300000e+07,1.839653e+08,18.577425,7.381750,5249.250000
max,1.777808e+06,254.000000,6.588000e+08,2.925500e+09,623.394200,10.000000,41375.000000


**What to notice**: lots of films have `budget = 0` and `revenue = 0` (unknown), and some films have no `overview`. We handle this in the next step.

## 3. Cleaning

In [8]:
# 1. Remove duplicates
df = df.drop_duplicates(subset="movie_id")
df = df.drop_duplicates(subset=["title", "release_date"])

# 2. Convert the date (text -> real date). Invalid dates become NaT (missing).
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

# 3. Remove films without title or date, and films not released yet
df = df.dropna(subset=["title", "release_date"])
df = df[df["release_date"] <= pd.Timestamp.today()]

# 4. Empty overview -> missing value
df["overview"] = df["overview"].str.strip()
df["overview"] = df["overview"].replace("", np.nan)

# 5. Zeros that really mean "unknown" -> missing value
for column in ["budget", "revenue", "runtime"]:
    df[column] = df[column].replace(0, np.nan)

# 6. A rating without any vote is not a real rating
df.loc[df["vote_count"] == 0, "vote_average"] = np.nan

# 7. Add the release year (useful later)
df["release_year"] = df["release_date"].dt.year

df = df.reset_index(drop=True)
print("Shape after cleaning:", df.shape)
print(df.isna().sum())

Shape after cleaning: (2972, 14)
movie_id               0
title                  0
overview              11
release_date           0
runtime               23
original_language      0
genres                 0
keywords               0
budget               588
revenue              536
popularity             0
vote_average          24
vote_count             0
release_year           0
dtype: int64


**Important**: we do **not** fill the missing values (with the mean, for example) now. We will do it later inside the ML pipeline, only with information from the training set. Otherwise the model would "see" information from the test set → this is called **data leakage**.

## 4. Separate the variable types

In [9]:
numeric_columns = ["runtime", "budget", "revenue", "popularity", "vote_average", "vote_count"]
categorical_columns = ["original_language", "genres"]
text_columns = ["title", "overview", "keywords"]
date_columns = ["release_date"]

print("Numeric    :", numeric_columns)
print("Categorical:", categorical_columns)
print("Text       :", text_columns)
print("Date       :", date_columns)

Numeric    : ['runtime', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count']
Categorical: ['original_language', 'genres']
Text       : ['title', 'overview', 'keywords']
Date       : ['release_date']


## 5. Save the clean data as CSV

In [10]:
# A CSV cannot store a Python list, so we write ["Action", "Comedy"] as "Action|Comedy"
def list_to_text(items):
    return "|".join(items)

df_csv = df.copy()
df_csv["genres"] = df_csv["genres"].apply(list_to_text)
df_csv["keywords"] = df_csv["keywords"].apply(list_to_text)

os.makedirs("data/processed", exist_ok=True)
df_csv.to_csv("data/processed/movies_clean.csv", index=False)
print("Saved:", df_csv.shape)

Saved: (2972, 14)


## 6. MongoDB

**What is MongoDB?** A *document* database: each film is stored as a document (like a JSON object). It is flexible and a list such as `genres` can be stored as it is.

**Before running**: MongoDB must be running. With Docker:
`docker run -d -p 27017:27017 --name mongo mongo:7`

In [11]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=3000)
client.admin.command("ping")                         # fails if MongoDB is not running
print("Connected to MongoDB")

collection = client["movie_intelligence"]["movies"]  # database -> collection

Connected to MongoDB


In [12]:
# Prepare the documents: MongoDB does not accept NaN, it needs None
df_mongo = df.copy()
df_mongo["release_date"] = df_mongo["release_date"].dt.strftime("%Y-%m-%d")   # date as text
df_mongo = df_mongo.astype(object).where(df_mongo.notna(), None)
records = df_mongo.to_dict("records")

collection.delete_many({})          # empty the collection so we can re-run the notebook safely
collection.insert_many(records)
print("Films in MongoDB:", collection.count_documents({}))

Films in MongoDB: 2972


### MongoDB queries (`find`)

In [13]:
# Query 1: best rated films with many votes
print("Query 1 - rating >= 8 and votes >= 1000")
best = collection.find(
    {"vote_average": {"$gte": 8}, "vote_count": {"$gte": 1000}},   # filter
    {"_id": 0, "title": 1, "vote_average": 1, "vote_count": 1}     # fields to show
).sort("vote_average", -1).limit(5)
for film in best:
    print(film)

# Query 2: comedies shorter than 90 minutes
count = collection.count_documents({"genres": "Comedy", "runtime": {"$lt": 90}})
print("\nQuery 2 - comedies under 90 min:", count)

# Query 3: French films released since 2015
count = collection.count_documents({"original_language": "fr", "release_year": {"$gte": 2015}})
print("Query 3 - French films since 2015:", count)

# Query 4: titles starting with "Star"
print("\nQuery 4 - titles starting with Star")
for film in collection.find({"title": {"$regex": "^Star"}}, {"_id": 0, "title": 1}).limit(5):
    print(film["title"])

Query 1 - rating >= 8 and votes >= 1000
{'title': 'Avatar Aang: The Last Airbender', 'vote_average': 9.128, 'vote_count': 1258}
{'title': 'Swapped', 'vote_average': 8.863, 'vote_count': 2224}
{'title': 'Demon Slayer: Kimetsu no Yaiba Infinity Castle', 'vote_average': 8.796, 'vote_count': 2239}
{'title': 'The Shawshank Redemption', 'vote_average': 8.732, 'vote_count': 31494}
{'title': 'The Godfather', 'vote_average': 8.7, 'vote_count': 23683}

Query 2 - comedies under 90 min: 154
Query 3 - French films since 2015: 17

Query 4 - titles starting with Star
Star Wars
Star Wars: The Last Jedi
Star Wars: The Force Awakens
Star Wars: The Rise of Skywalker
Star Wars: Episode I - The Phantom Menace


### MongoDB aggregations
An aggregation is a **pipeline**: a list of stages, where the result of one stage goes to the next.

| Stage | Meaning |
|---|---|
| `$match` | keep only the documents that match a condition (like WHERE) |
| `$unwind` | split a list: a film with 3 genres becomes 3 documents |
| `$group` | group documents and compute (count, average...) (like GROUP BY) |
| `$sort` | sort the result |
| `$project` | choose / rename the fields to show |

In [14]:
# Aggregation 1: average rating and number of films per genre
pipeline_genre = [
    {"$match": {"vote_count": {"$gte": 100}}},                  # films with enough votes
    {"$unwind": "$genres"},                                     # one row per (film, genre)
    {"$group": {"_id": "$genres",
                "n_films": {"$sum": 1},
                "avg_rating": {"$avg": "$vote_average"}}},
    {"$sort": {"avg_rating": -1}},
    {"$project": {"_id": 0, "genre": "$_id", "n_films": 1,
                  "avg_rating": {"$round": ["$avg_rating", 2]}}},
]
result = list(collection.aggregate(pipeline_genre))
pd.DataFrame(result)

,n_films,genre,avg_rating
0,3,Documentary,7.54
1,76,War,7.35
2,111,History,7.29
3,39,Western,7.19
4,931,Drama,7.18
5,51,Music,7.18
6,340,Animation,7.16
7,374,Romance,6.93
8,418,Crime,6.89
9,719,Adventure,6.85


In [15]:
# Aggregation 2: average number of votes per release year (since 2015)
pipeline_year = [
    {"$match": {"release_year": {"$gte": 2015}}},
    {"$group": {"_id": "$release_year",
                "n_films": {"$sum": 1},
                "avg_votes": {"$avg": "$vote_count"}}},
    {"$sort": {"_id": 1}},
    {"$project": {"_id": 0, "year": "$_id", "n_films": 1,
                  "avg_votes": {"$round": ["$avg_votes", 0]}}},
]
result = list(collection.aggregate(pipeline_year))
pd.DataFrame(result)

,n_films,year,avg_votes
0,66,2015,7097.0
1,82,2016,6796.0
2,92,2017,7065.0
3,76,2018,6231.0
4,89,2019,4985.0
5,61,2020,3060.0
6,95,2021,3668.0
7,105,2022,3187.0
8,114,2023,2407.0
9,129,2024,1648.0


## What to remember (for the oral)
- We **never fill** missing values here → avoids data leakage.
- `budget = 0` means unknown in TMDB → replaced by `NaN`.
- MongoDB stores one document per film; `$unwind` is needed to count by genre because `genres` is a list.
- The clean CSV (`data/processed/movies_clean.csv`) is the input of the next notebooks.